In [0]:
from pyspark.sql.functions import col, current_timestamp

# Read from silver layer
df_silver = spark.table("ecommerce.e_comm_silver.products")
df_silver.createOrReplaceTempView("vw_silver")
# Transform for gold layer:
# 1. Filter only valid records
# 2. Select business columns only
# 3. Rename for business clarity
df_gold_dim = spark.sql("""
        select 
            product_key,
            product_id,
            product_name,
            category as product_category,
            brand as product_brand,
            price as product_price,
            current_timestamp() as load_ts
        from vw_silver
        where dq_note ="is_valid"
    """)

# Display sample
print(f"Total records in gold dimension: {df_gold_dim.count()}")

# Write to gold layer
df_gold_dim.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("ecommerce.e_comm_gold.dimProducts")

print("✅ Gold dimension table ecommerce.e_comm_gold.dimProducts created successfully")